<a href="https://colab.research.google.com/github/JamshedAli18/LLm-finetuning/blob/main/LoraFit_SmolLM2_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install everything**

In [ ]:
!pip install -q transformers datasets trl peft bitsandbytes accelerate

**Pick your dataset**

*   The best beginner dataset is Alpaca (tatsu-lab/alpaca).
*   It has 52K instruction-following examples already in the exact format SmolLM2 expects.



In [ ]:
from datasets import load_dataset

# Load the Alpaca dataset
dataset = load_dataset("tatsu-lab/alpaca", split="train")



In [ ]:
# Preview what it looks like
dataset[3]

**Format into SmolLM2's chat template**

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def format_alpaca_to_chat(example):
    """Convert an Alpaca row into SmolLM2's ChatML format."""
    # Build the user message
    if example["input"]:
        user_msg = f"{example['instruction']}\n\n{example['input']}"
    else:
        user_msg = example["instruction"]

    # Apply the model's native chat template
    messages = [
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": example["output"]},
    ]
    # tokenizer.apply_chat_template renders it to a string
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,          # return string, not token ids
        add_generation_prompt=False
    )
    return {"text": text}

# Apply to the full dataset
dataset = dataset.map(format_alpaca_to_chat)

# See what the formatted text looks like
print(dataset[0]["text"])

In [ ]:
dataset = dataset.select(range(1000)).train_test_split(test_size=0.1)
print(f"Train: {len(dataset['train'])}  |  Val: {len(dataset['test'])}")

**Load model in 4-bit (QLoRA)**

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config (the "Q" in QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best for LLM weights
    bnb_4bit_compute_dtype=torch.float16, # compute in fp16, store in 4-bit
    bnb_4bit_use_double_quant=True,       # double quantization saves ~0.4 bits/param
)

# Load the model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",         # puts it on GPU automatically
    trust_remote_code=True,
)

# Prepare frozen layers for gradient checkpointing (saves memory)
model = prepare_model_for_kbit_training(model)

# LoRA config — which layers to add adapters to
lora_config = LoraConfig(
    r=16,                      # rank: 16 is a good default for 135M model
    lora_alpha=32,             # scaling factor α (rule of thumb: 2×r)
    target_modules=[           # which weight matrices to add adapters to
        "q_proj", "k_proj",    # attention query and key
        "v_proj", "o_proj",    # attention value and output
        "gate_proj", "up_proj", "down_proj",  # MLP layers
    ],
    lora_dropout=0.05,         # small dropout to prevent overfitting
    bias="none",               # don't train bias terms
    task_type="CAUSAL_LM",     # language modeling task
)

# Wrap the model with LoRA adapters
model = get_peft_model(model, lora_config)

# See how many params are trainable
model.print_trainable_parameters()
# Output: trainable params: ~1.3M || all params: ~136M || trainable: ~0.97%

**Train with SFTTrainer**

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./smollm2-alpaca-lora",

    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,

    fp16=False,   # <-- turned off
    bf16=True,    # <-- turned on

    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,

    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

trainer.train()

 **Save the LoRA adapter**

In [ ]:
# Save just the LoRA adapter weights (very small — ~10MB)
trainer.save_model("./smollm2-alpaca-lora/final-adapter")
tokenizer.save_pretrained("./smollm2-alpaca-lora/final-adapter")

print("Adapter saved!")

# **Inference**

In [ ]:
# Disable gradient checkpointing for inference
model.config.use_cache = True
model.gradient_checkpointing_disable()
model.eval()

def chat(instruction):
    messages = [{"role": "user", "content": instruction}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (skip the prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Test it!
print(chat("Give me 3 tips for staying productive."))
print("---")
print(chat("Explain black holes simply."))
print("---")
print(chat("Write a short poem about rain."))

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!")